In [8]:
import cv2
import mediapipe as mp
import numpy as np
import math
import time
import os

In [26]:
import urllib.request

url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"

urllib.request.urlretrieve(
    url,
    "hand_landmarker.task"
)

print("Downloaded successfully!")

Downloaded successfully!


In [27]:
MODEL_PATH = "hand_landmarker.task"

CAMERA_INDEX = 0

FRAME_WIDTH = 1280
FRAME_HEIGHT = 720

MAX_HANDS = 2

CONTACT_DISTANCE = 45

In [28]:
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

In [29]:
if not os.path.exists(MODEL_PATH):

    raise FileNotFoundError(
        f"\nModel not found:\n{MODEL_PATH}\n\n"
        "Please put hand_landmarker.task next to this Python file."
    )

In [30]:

# =========================================================
# HAND LANDMARK INDEX
# =========================================================

WRIST = 0

THUMB_TIP = 4
INDEX_TIP = 8
MIDDLE_TIP = 12
RING_TIP = 16
PINKY_TIP = 20

INDEX_PIP = 6
MIDDLE_PIP = 10
RING_PIP = 14
PINKY_PIP = 18


FINGERTIPS = {
    "thumb": THUMB_TIP,
    "index": INDEX_TIP,
    "middle": MIDDLE_TIP,
    "ring": RING_TIP,
    "pinky": PINKY_TIP
}


# =========================================================
# COLORS
# =========================================================

BLUE = (255, 170, 30)

LIGHT_BLUE = (255, 230, 130)

GLOW_BLUE = (255, 100, 0)

WHITE = (255, 255, 255)

In [31]:
# =========================================================
# UTILITY
# =========================================================

def distance(p1, p2):

    return math.hypot(
        p1[0] - p2[0],
        p1[1] - p2[1]
    )


def midpoint(p1, p2):

    return (
        int((p1[0] + p2[0]) / 2),
        int((p1[1] + p2[1]) / 2)
    )


def rotate_point(point, center, angle):

    x, y = point

    cx, cy = center

    cos_a = math.cos(angle)
    sin_a = math.sin(angle)

    nx = (
        cos_a * (x - cx)
        -
        sin_a * (y - cy)
        +
        cx
    )

    ny = (
        sin_a * (x - cx)
        +
        cos_a * (y - cy)
        +
        cy
    )

    return int(nx), int(ny)


def polygon_center(points):

    if not points:
        return 0, 0

    x = sum(p[0] for p in points) / len(points)

    y = sum(p[1] for p in points) / len(points)

    return int(x), int(y)


In [32]:
# =========================================================
# HAND CONVERSION
# =========================================================

def landmarks_to_points(hand_landmarks, width, height):

    points = []

    for landmark in hand_landmarks:

        x = int(
            landmark.x * width
        )

        y = int(
            landmark.y * height
        )

        points.append(
            (x, y)
        )

    return points



In [33]:
# =========================================================
# FINGER STATE
# =========================================================

def finger_is_open(
    points,
    tip,
    pip
):

    wrist = points[WRIST]

    tip_distance = distance(
        points[tip],
        wrist
    )

    pip_distance = distance(
        points[pip],
        wrist
    )

    return tip_distance > pip_distance * 1.15


def is_fist(points):

    fingers = [

        (INDEX_TIP, INDEX_PIP),

        (MIDDLE_TIP, MIDDLE_PIP),

        (RING_TIP, RING_PIP),

        (PINKY_TIP, PINKY_PIP)
    ]

    closed = 0

    for tip, pip in fingers:

        if not finger_is_open(
            points,
            tip,
            pip
        ):

            closed += 1

    return closed >= 3


def is_open_hand(points):

    fingers = [

        (INDEX_TIP, INDEX_PIP),

        (MIDDLE_TIP, MIDDLE_PIP),

        (RING_TIP, RING_PIP),

        (PINKY_TIP, PINKY_PIP)
    ]

    opened = 0

    for tip, pip in fingers:

        if finger_is_open(
            points,
            tip,
            pip
        ):

            opened += 1

    return opened >= 3



In [34]:

# =========================================================
# FINGER CONTACT
# =========================================================

def detect_contacts(
    hand1,
    hand2
):

    contacts = []

    for name1, id1 in FINGERTIPS.items():

        p1 = hand1[id1]

        for name2, id2 in FINGERTIPS.items():

            p2 = hand2[id2]

            d = distance(
                p1,
                p2
            )

            if d <= CONTACT_DISTANCE:

                contact = midpoint(
                    p1,
                    p2
                )

                contacts.append({
                    "left": name1,
                    "right": name2,
                    "point": contact
                })

    return contacts


In [35]:


# =========================================================
# GLASS SHAPE
# =========================================================

class GlassShape:

    def __init__(
        self,
        contacts
    ):

        self.contacts = contacts

        self.center = polygon_center(
            contacts
        )

        self.scale = 1.0

        self.rotation = 0.0

        self.locked = False

        self.selected = False

        self.created_at = time.time()

    # -----------------------------------------------------

    def update_contacts(
        self,
        contacts
    ):

        if self.locked:
            return

        if not contacts:
            return

        self.contacts = contacts

        self.center = polygon_center(
            contacts
        )

    # -----------------------------------------------------

    def move(
        self,
        dx,
        dy
    ):

        self.center = (
            self.center[0] + dx,
            self.center[1] + dy
        )

    # -----------------------------------------------------

    def resize(
        self,
        amount
    ):

        self.scale *= amount

        self.scale = max(
            0.25,
            min(
                self.scale,
                5.0
            )
        )

    # -----------------------------------------------------

    def rotate(
        self,
        angle
    ):

        self.rotation += angle

    # -----------------------------------------------------

    def get_vertices(self):

        count = len(
            self.contacts
        )

        if count == 0:

            return []

        cx, cy = self.center

        # =================================================
        # TWO CONTACTS
        # =================================================

        if count == 2:

            p1 = self.contacts[0]

            p2 = self.contacts[1]

            dx = p2[0] - p1[0]

            dy = p2[1] - p1[1]

            length = max(
                math.hypot(
                    dx,
                    dy
                ),
                1
            )

            nx = -dy / length

            ny = dx / length

            width = (
                35 *
                self.scale
            )

            vertices = [

                (
                    int(
                        p1[0] +
                        nx * width
                    ),
                    int(
                        p1[1] +
                        ny * width
                    )
                ),

                (
                    int(
                        p2[0] +
                        nx * width
                    ),
                    int(
                        p2[1] +
                        ny * width
                    )
                ),

                (
                    int(
                        p2[0] -
                        nx * width
                    ),
                    int(
                        p2[1] -
                        ny * width
                    )
                ),

                (
                    int(
                        p1[0] -
                        nx * width
                    ),
                    int(
                        p1[1] -
                        ny * width
                    )
                )
            ]

        # =================================================
        # 3+
        # =================================================

        else:

            vertices = []

            for p in self.contacts:

                dx = (
                    p[0] - cx
                )

                dy = (
                    p[1] - cy
                )

                vertices.append(

                    (
                        int(
                            cx +
                            dx *
                            self.scale
                        ),

                        int(
                            cy +
                            dy *
                            self.scale
                        )
                    )
                )

        # =================================================
        # ROTATION
        # =================================================

        result = []

        for point in vertices:

            result.append(
                rotate_point(
                    point,
                    self.center,
                    self.rotation
                )
            )

        return result

    # -----------------------------------------------------

    def contains(
        self,
        point
    ):

        vertices = self.get_vertices()

        if len(vertices) < 3:
            return False

        polygon = np.array(
            vertices,
            dtype=np.int32
        )

        result = cv2.pointPolygonTest(
            polygon,
            point,
            False
        )

        return result >= 0



In [36]:

# =========================================================
# DRAW GLASS SHAPE
# =========================================================

def draw_glass_shape(
    frame,
    shape
):

    vertices = shape.get_vertices()

    if len(vertices) < 3:
        return

    polygon = np.array(
        vertices,
        dtype=np.int32
    )

    # =====================================================
    # GLASS BODY
    # =====================================================

    overlay = frame.copy()

    cv2.fillPoly(
        overlay,
        [polygon],
        BLUE
    )

    frame[:] = cv2.addWeighted(
        overlay,
        0.18,
        frame,
        0.82,
        0
    )

    # =====================================================
    # GLOW
    # =====================================================

    glow = np.zeros_like(
        frame
    )

    thickness = 10

    if shape.selected:

        thickness = 15

    cv2.polylines(
        glow,
        [polygon],
        True,
        GLOW_BLUE,
        thickness,
        cv2.LINE_AA
    )

    glow = cv2.GaussianBlur(
        glow,
        (0, 0),
        12
    )

    frame[:] = cv2.addWeighted(
        frame,
        1.0,
        glow,
        0.5,
        0
    )

    # =====================================================
    # BORDER
    # =====================================================

    border_width = 3

    if shape.selected:

        border_width = 5

    cv2.polylines(
        frame,
        [polygon],
        True,
        LIGHT_BLUE,
        border_width,
        cv2.LINE_AA
    )

    # =====================================================
    # INNER GLASS LINES
    # =====================================================

    cx, cy = shape.center

    for point in vertices:

        ix = int(
            cx +
            (point[0] - cx)
            * 0.75
        )

        iy = int(
            cy +
            (point[1] - cy)
            * 0.75
        )

        cv2.line(
            frame,
            (cx, cy),
            (ix, iy),
            (255, 200, 100),
            1,
            cv2.LINE_AA
        )



In [37]:
# =========================================================
# DRAW HAND
# =========================================================

def draw_hand(
    frame,
    points
):

    connections = [

        (0, 1),
        (1, 2),
        (2, 3),
        (3, 4),

        (0, 5),
        (5, 6),
        (6, 7),
        (7, 8),

        (9, 10),
        (10, 11),
        (11, 12),

        (13, 14),
        (14, 15),
        (15, 16),

        (17, 18),
        (18, 19),
        (19, 20),

        (5, 9),
        (9, 13),
        (13, 17),
        (0, 17)
    ]

    for a, b in connections:

        cv2.line(
            frame,
            points[a],
            points[b],
            (120, 180, 255),
            2,
            cv2.LINE_AA
        )

    for point in points:

        cv2.circle(
            frame,
            point,
            4,
            (255, 220, 100),
            -1,
            cv2.LINE_AA
        )



In [38]:
# =========================================================
# CREATE MEDIAPIPE
# =========================================================

options = HandLandmarkerOptions(

    base_options=BaseOptions(
        model_asset_path=MODEL_PATH
    ),

    running_mode=RunningMode.VIDEO,

    num_hands=MAX_HANDS,

    min_hand_detection_confidence=0.65,

    min_hand_presence_confidence=0.65,

    min_tracking_confidence=0.65
)



In [39]:
# =========================================================
# CAMERA
# =========================================================

cap = cv2.VideoCapture(
    CAMERA_INDEX
)

cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    FRAME_WIDTH
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    FRAME_HEIGHT
)


if not cap.isOpened():

    raise RuntimeError(
        "Camera could not be opened."
    )


# =========================================================
# STATES
# =========================================================

shapes = []

active_creation_shape = None

selected_shape = None

previous_drag_position = None

previous_scale_distance = None

previous_rotation_angle = None

timestamp_ms = 0



In [40]:
# =========================================================
# MAIN
# =========================================================

with HandLandmarker.create_from_options(
    options
) as landmarker:

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.flip(
            frame,
            1
        )

        height, width = frame.shape[:2]

        # =================================================
        # MEDIAPIPE IMAGE
        # =================================================

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        timestamp_ms += 33

        result = landmarker.detect_for_video(
            mp_image,
            timestamp_ms
        )

        hands = []

        for hand_landmarks in result.hand_landmarks:

            points = landmarks_to_points(
                hand_landmarks,
                width,
                height
            )

            hands.append(
                points
            )

            draw_hand(
                frame,
                points
            )

        # =================================================
        # TWO HANDS
        # =================================================

        if len(hands) == 2:

            hand1 = hands[0]

            hand2 = hands[1]

            fist1 = is_fist(hand1)

            fist2 = is_fist(hand2)

            open1 = is_open_hand(hand1)

            open2 = is_open_hand(hand2)

            # =================================================
            # CONTACT
            # =================================================

            contact_data = detect_contacts(
                hand1,
                hand2
            )

            contact_points = [

                item["point"]

                for item in contact_data
            ]

            # =================================================
            # CREATE / UPDATE SHAPE
            # =================================================

            if contact_points:

                if active_creation_shape is None:

                    active_creation_shape = GlassShape(
                        contact_points
                    )

                    shapes.append(
                        active_creation_shape
                    )

                else:

                    active_creation_shape.update_contacts(
                        contact_points
                    )

            # =================================================
            # SHOW CONTACT POINTS
            # =================================================

            for contact in contact_data:

                p = contact["point"]

                cv2.circle(
                    frame,
                    p,
                    8,
                    (255, 230, 100),
                    -1,
                    cv2.LINE_AA
                )

                cv2.circle(
                    frame,
                    p,
                    18,
                    (255, 100, 0),
                    2,
                    cv2.LINE_AA
                )

            # =================================================
            # BOTH FISTS
            # =================================================

            if fist1 and fist2:

                # First priority:
                # scale existing shape

                c1 = hand1[WRIST]

                c2 = hand2[WRIST]

                center_between = midpoint(
                    c1,
                    c2
                )

                target = None

                for shape in reversed(shapes):

                    if shape.contains(
                        center_between
                    ):

                        target = shape

                        break

                if target is not None:

                    selected_shape = target

                    target.selected = True

                    current_distance = distance(
                        c1,
                        c2
                    )

                    if previous_scale_distance is not None:

                        ratio = (
                            current_distance /
                            previous_scale_distance
                        )

                        target.resize(
                            ratio
                        )

                    previous_scale_distance = (
                        current_distance
                    )

                # Lock shape being created

                if active_creation_shape is not None:

                    active_creation_shape.locked = True

                    active_creation_shape.selected = False

                    active_creation_shape = None

            else:

                previous_scale_distance = None

            # =================================================
            # ONE FIST + ONE OPEN
            # =================================================

            if (
                fist1 and open2
            ) or (
                fist2 and open1
            ):

                fist_hand = (
                    hand1
                    if fist1
                    else hand2
                )

                open_hand = (
                    hand2
                    if fist1
                    else hand1
                )

                fist_center = fist_hand[WRIST]

                # ---------------------------------------------
                # Find selected shape
                # ---------------------------------------------

                if selected_shape is None:

                    for shape in reversed(shapes):

                        if shape.contains(
                            fist_center
                        ):

                            selected_shape = shape

                            shape.selected = True

                            break

                # ---------------------------------------------
                # Rotation
                # ---------------------------------------------

                if selected_shape is not None:

                    open_center = open_hand[WRIST]

                    cx, cy = selected_shape.center

                    angle = math.atan2(
                        open_center[1] - cy,
                        open_center[0] - cx
                    )

                    if previous_rotation_angle is not None:

                        delta = (
                            angle -
                            previous_rotation_angle
                        )

                        if delta > math.pi:

                            delta -= 2 * math.pi

                        elif delta < -math.pi:

                            delta += 2 * math.pi

                        selected_shape.rotate(
                            delta
                        )

                    previous_rotation_angle = angle

            else:

                previous_rotation_angle = None

        # =================================================
        # ONE HAND
        # =================================================

        elif len(hands) == 1:

            hand = hands[0]

            fist = is_fist(
                hand
            )

            opened = is_open_hand(
                hand
            )

            center = hand[WRIST]

            # =================================================
            # FIST -> SELECT / MOVE
            # =================================================

            if fist:

                if selected_shape is None:

                    for shape in reversed(shapes):

                        if shape.contains(
                            center
                        ):

                            selected_shape = shape

                            shape.selected = True

                            previous_drag_position = center

                            break

                else:

                    if previous_drag_position is not None:

                        dx = (
                            center[0]
                            -
                            previous_drag_position[0]
                        )

                        dy = (
                            center[1]
                            -
                            previous_drag_position[1]
                        )

                        selected_shape.move(
                            dx,
                            dy
                        )

                    previous_drag_position = center

            # =================================================
            # OPEN -> RELEASE
            # =================================================

            elif opened:

                if selected_shape is not None:

                    selected_shape.selected = False

                selected_shape = None

                previous_drag_position = None

        else:

            previous_drag_position = None

        # =================================================
        # DRAW SHAPES
        # =================================================

        for shape in shapes:

            draw_glass_shape(
                frame,
                shape
            )

        # =================================================
        # UI
        # =================================================

        cv2.putText(
            frame,
            "HAND GLASS ENGINE",
            (25, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.85,
            LIGHT_BLUE,
            2,
            cv2.LINE_AA
        )

        cv2.putText(
            frame,
            "Q = EXIT",
            (25, height - 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            WHITE,
            1,
            cv2.LINE_AA
        )

        # =================================================
        # SHOW
        # =================================================

        cv2.imshow(
            "Hand Glass Engine",
            frame
        )

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):

            break

# =========================================================
# CLEANUP
# =========================================================

cap.release()

cv2.destroyAllWindows()